# STEP4. Olist 분석 Q&A 챗봇 만들기 (RAG)

STEP1에서 얻은 **분석 결과**를 책장에 꽂아 두고, 질문이 오면 **찾아보고 근거 ID와 함께 답하는** 챗봇을 만듭니다.

| 채널 | 책장(자료) | 답할 수 있는 것 |
|---|---|---|
| **A. 분석 사실 확인** | 사실표 `olist_facts.csv` (35행) | 분석 숫자 (주문·배송·리뷰·판매자) |
| **B. 분석 + 모델 안내** | 사실표 + 모델 설명서 `olist_model_guide.md` | 분석 숫자 + 모델이 하는 일·입력·한계 |
| **C. 분석 + 모델 + 규정** | 위 둘 + 규정집 PDF `olist_seller_policy.pdf` | 위 내용 + 판매자 규정 (출고 기한·배송 보상·등급별 조치) |

같은 질문을 세 채널에 던져 보면 **책장에 무엇을 넣었느냐에 따라 답이 달라지는 것**을 확인할 수 있습니다.

```
준비 (한 번)   자료 → 조각(Document) → 임베딩(좌표) → 벡터 저장소
질문 (매번)    질문 → 비슷한 조각 4개 찾기 → [근거]+질문을 AI에게 → 답 + [근거 ID] → ID 점검
```

## 준비
- Colab 왼쪽 🔑(보안 비밀)에 `OPENAI_API_KEY` 등록 → 노트북 액세스 켜기
- Drive `Olist/rag` 폴더에 `olist_facts.csv`, `olist_model_guide.md`, `olist_seller_policy.pdf`

> 규정집 PDF는 RAG 실습을 위해 만든 가상의 규정입니다. 실제 Olist의 정책이 아닙니다.

In [4]:
# 셀 1. 설치 (LangChain 부품)
!pip install -q langchain-core langchain-openai langchain-community pypdf
%pip install "requests==2.32.4"
%pip check

  Using cached requests-2.32.4-py3-none-any.whl.metadata (4.9 kB)
Using cached requests-2.32.4-py3-none-any.whl (64 kB)
  Attempting uninstall: requests
    Found existing installation: requests 2.34.2
    Uninstalling requests-2.34.2:
      Successfully uninstalled requests-2.34.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-community 0.4.2 requires requests<3.0.0,>=2.32.5, but you have requests 2.32.4 which is incompatible.
ipython 7.34.0 requires jedi, which is not installed.
langchain-community 0.4.2 has requirement requests<3.0.0,>=2.32.5, but you have requests 2.32.4.


In [1]:
%pip install "requests==2.32.4" "langchain-community==0.3.27"
%pip check

ipython 7.34.0 requires jedi, which is not installed.
langchain-openai 1.7.4 has requirement langchain-core<2.0.0,>=1.7.5, but you have langchain-core 0.3.86.
langchain-classic 1.0.8 has requirement langchain-core<2.0.0,>=1.4.4, but you have langchain-core 0.3.86.
langchain-classic 1.0.8 has requirement langchain-text-splitters<2.0.0,>=1.1.2, but you have langchain-text-splitters 0.3.11.
langgraph 1.2.18 has requirement langchain-core<2,>=1.4.7, but you have langchain-core 0.3.86.
langgraph-prebuilt 1.1.1 has requirement langchain-core>=1.3.1, but you have langchain-core 0.3.86.
langgraph-sdk 0.4.10 has requirement langchain-core<2,>=1.4.0, but you have langchain-core 0.3.86.


In [7]:
%pip install "requests==2.32.4" "langchain-community==0.3.27" "langchain-openai==0.3.35" "langchain-core>=0.3.78,<1.0.0"
%pip check

  Attempting uninstall: langchain-openai
    Found existing installation: langchain-openai 1.7.4
    Uninstalling langchain-openai-1.7.4:
      Successfully uninstalled langchain-openai-1.7.4
ipython 7.34.0 requires jedi, which is not installed.
langchain-classic 1.0.8 has requirement langchain-core<2.0.0,>=1.4.4, but you have langchain-core 0.3.86.
langchain-classic 1.0.8 has requirement langchain-text-splitters<2.0.0,>=1.1.2, but you have langchain-text-splitters 0.3.11.
langgraph 1.2.18 has requirement langchain-core<2,>=1.4.7, but you have langchain-core 0.3.86.
langgraph-prebuilt 1.1.1 has requirement langchain-core>=1.3.1, but you have langchain-core 0.3.86.
langgraph-sdk 0.4.10 has requirement langchain-core<2,>=1.4.0, but you have langchain-core 0.3.86.


In [2]:
# 셀 2. Drive 연결 + 경로 + API 키
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, re
import pandas as pd

base_path = "/content/drive/MyDrive/Colab Notebooks/Olist"
rag_path  = f"{base_path}/rag"                         # 사실표 · 모델 설명서 · 규정집 위치
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
print(os.listdir(rag_path))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['olist_facts.csv', 'olist_model_guide.md', 'olist_seller_policy.pdf']


## ① 책장 1 : 사실표

사실표 = 분석에서 확인한 사실을 **한 줄에 하나씩** 적은 표입니다. 기자가 기사를 쓰기 전에 만드는 '팩트 시트'와 같습니다.

| 열 | 뜻 | 예 |
|---|---|---|
| 사실ID | 답에 붙일 번호 | F14 |
| 분야 | 주문 · 배송 · 리뷰 · 판매자 … | 리뷰 |
| 내용 | 사실 한 문장 (숫자 포함) | 제때 도착 4.30점, 늦게 도착 2.57점 |
| 기준 | 어떤 데이터·기간으로 계산했나 | 배송 완료 주문의 리뷰 |
| 출처 | 어느 노트북에서 나왔나 | 05_eda |

In [3]:
# 셀 3. 사실표 읽기 → 조각(Document) 만들기 : 사실 1행 = 조각 1개
from langchain_core.documents import Document

facts = pd.read_csv(f"{rag_path}/olist_facts.csv")
display(facts.head())

fact_docs = [Document(page_content=f"{r['내용']} (기준: {r['기준']})",
                      metadata={"id": r["사실ID"], "출처": r["출처"]})
             for _, r in facts.iterrows()]
print(f"사실 조각 : {len(fact_docs)}개")
print(fact_docs[13].metadata, fact_docs[13].page_content)

,사실ID,분야,내용,기준,출처
0,F01,주문,"분석 기준 주문은 96,203건이다.","배송 완료 주문, 2017-01-05 ~ 2018-08-29",05_eda
1,F02,주문,"분석 기준 매출 합계는 R$ 15,374,497(약 R$ 15.37M)이다.",배송 완료 주문의 결제 금액 합계,05_eda
2,F03,주문,객단가(주문 1건 평균 결제 금액)는 R$ 159.8이다.,배송 완료 주문,05_eda
3,F04,주문,평균보다 작은 금액의 주문이 70.4%다. 소수의 큰 주문이 평균을 끌어올린다.,배송 완료 주문,05_eda
4,F05,주문,"주문이 가장 많았던 날은 2017-11-24(블랙프라이데이)로 1,147건이다. 다...",날짜별 주문 수,05_eda


사실 조각 : 35개
{'id': 'F14', '출처': '05_eda'} 제때 도착한 주문의 평균 리뷰는 4.30점, 늦게 도착한 주문은 2.57점이다. (기준: 배송 완료 주문의 리뷰)


## ② 책장 2 : 모델 설명서

모델 설명서는 `## M09 판매자 이탈 모델` 처럼 **제목 하나 = 모델 하나**로 써 두었습니다.
그래서 글자 수로 자르지 않고 **제목 단위로** 자릅니다. (공약집을 '공약 항목 하나'씩 자르는 것과 같은 생각)

In [4]:
# 셀 4. 모델 설명서 읽기 → '## 제목' 하나 = 조각 1개
text = open(f"{rag_path}/olist_model_guide.md", encoding="utf-8").read()

guide_docs = []
for block in text.split("\n## ")[1:]:                 # 맨 앞(# 제목)은 빼고
    title, body = block.split("\n", 1)
    guide_docs.append(Document(page_content=f"{title}\n{body.strip()}",
                               metadata={"id": title.split()[0], "출처": "모델 설명서"}))

for d in guide_docs:
    print(d.metadata["id"], "|", d.page_content.split("\n")[0], f"({len(d.page_content)}자)")

M00 | M00 모델 파일 한눈에 보기 (456자)
M06 | M06 상품 추천 모델 (part6_recommend.pkl) (382자)
M71 | M71 리뷰 감성 머신러닝 모델 (part7_1_sentiment_ml.pkl) (337자)
M72 | M72 리뷰 감성 딥러닝 모델 (part7_2) (242자)
M73 | M73 BERT 감성 분석 (07_3) (195자)
M08 | M08 판매자 유형 모델 (part8_seller_segment.pkl) (491자)
M09 | M09 판매자 이탈 모델 (part9_seller_churn.pkl) (640자)
M10 | M10 분석 결과를 전략으로 (424자)


## ②-2 책장 3 : 규정집 PDF

회사 규정집은 대부분 **PDF**입니다. PDF는 두 단계로 조각을 만듭니다.

| 단계 | 하는 일 | 도구 |
|---|---|---|
| ① 읽기 | 1쪽 = Document 1개, 쪽 번호(page)를 자동으로 저장 | `PyPDFLoader` |
| ② 자르기 | 글자 수가 아니라 **'제○조' 하나 = 조각 1개** | `re.split` |

조항 단위로 자르면 '제6조'가 두 조각으로 나뉘지 않아서 답이 정확해지고, 출처를 **'규정집 1쪽 제6조'**처럼 붙일 수 있습니다.

In [5]:
# 셀 4-2. 규정집 PDF 읽기 → '제○조' 하나 = 조각 1개
from langchain_community.document_loaders import PyPDFLoader

pages = PyPDFLoader(f"{rag_path}/olist_seller_policy.pdf").load()      # 1쪽 = Document 1개
print(f"PDF {len(pages)}쪽 · 1쪽 앞부분 ↓")
print(pages[0].page_content[:200])

policy_docs = []
for page in pages:
    for part in re.split(r"\n(?=제\d+조)", page.page_content):           # '제○조' 앞에서 자르기
        m = re.match(r"제(\d+)조", part)
        if not m:                                                        # 표지 · 머리말은 빼기
            continue
        lines = [l for l in part.splitlines() if not re.match(r"제\d+장|부칙", l)]  # 장 제목 줄 빼기
        policy_docs.append(Document(page_content="\n".join(lines).strip(),
                                    metadata={"id": f"R{int(m.group(1)):02d}",
                                              "출처": f"규정집 {page.metadata['page'] + 1}쪽"}))

print(f"\n규정 조각 : {len(policy_docs)}개")
print(policy_docs[5].metadata, policy_docs[5].page_content)

PDF 2쪽 · 1쪽 앞부분 ↓
Olist 판매자 운영 정책
실습용 가상 규정집 · 2018년 9월 시행
이 문서는 RAG 챗봇 실습을 위해 만든 가상의 규정입니다. 실제 Olist의 정책이 아닙니다.
제1장 총칙
제1조(목적)
이 규정은 Olist 마켓플레이스에 입점한 판매자의 출고·배송·고객 응대 기준과 판매자 등급별 관리 방법
을 정한다.
제2조(용어)
① “출고 기한”은 판매자가 

규정 조각 : 17개
{'id': 'R06', '출처': '규정집 1쪽'} 제6조(배송 지연 보상)
배송 예정일보다 3일 이상 늦게 도착한 주문의 고객이 요청하면 배송비를 환불한다. 비용은 지연 책임이
있는 쪽(판매자 또는 택배사)이 부담한다.


### PDF에서 주의할 점

| 문제 | 증상 | 해결 |
|---|---|---|
| 스캔한 PDF | 글자가 그림이라 `page_content`가 비어 있음 | OCR로 글자를 먼저 뽑기 |
| 표 | 칸이 줄글로 풀려 섞임 | 숫자가 중요한 표는 CSV로 따로 넣기 |
| 줄바꿈 | 문장 중간에 줄이 바뀜 (예: `주문\n할 때`) | 임베딩에는 큰 문제 없음. 보여 줄 때만 정리 |
| 조항이 쪽을 넘어감 | 한 조항이 두 쪽에 나뉨 | 쪽을 이어 붙인 뒤 '제○조'로 자르기 |

## ③ 임베딩 → 벡터 저장소 (채널마다 하나)

| 채널 | 넣는 조각 |
|---|---|
| A | 사실 35개 |
| B | 사실 35개 + 모델 설명 8개 |
| C | 사실 35개 + 모델 설명 8개 + 규정 17개 |

`InMemoryVectorStore` = 설치 없이 메모리에 만드는 벡터 저장소 (ChromaDB로 바꿔도 흐름은 같습니다)

In [8]:
# 셀 5. 세 채널의 벡터 저장소 만들기
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings

emb = OpenAIEmbeddings(model="text-embedding-3-small")
stores = {
    "A": InMemoryVectorStore.from_documents(fact_docs, emb),
    "B": InMemoryVectorStore.from_documents(fact_docs + guide_docs, emb),
    "C": InMemoryVectorStore.from_documents(fact_docs + guide_docs + policy_docs, emb),
}
print("벡터 저장소 준비 완료")

벡터 저장소 준비 완료


In [9]:
# 셀 6. 찾기만 먼저 보기 : 같은 질문, 채널마다 어떤 조각을 찾아오나?
def search(question, channel, k=4):
    for doc, score in stores[channel].similarity_search_with_score(question, k=k):
        print(f"  [{doc.metadata['id']}] 유사도 {score:.2f} | {doc.page_content[:50]}…")

q = "이탈 모델은 어떤 숫자를 보고 판단해?"
for ch in ["A", "B", "C"]:
    print(f"채널 {ch} :")
    search(q, ch)

채널 A :
  [F30] 유사도 0.52 | 이탈 모델 학습 대상은 2018-03~05에 판매한 판매자 1,567명이고, 이 중 377…
  [F31] 유사도 0.51 | 이탈 모델(로지스틱 회귀 + balanced)의 test 성능은 정밀도 0.43, 재현율 …
  [F21] 유사도 0.39 | 추천 모델 대상 상품은 2,498개(15개 카테고리)다. (기준: 서로 다른 주문 3건 이…
  [F25] 유사도 0.35 | 딥러닝(Embedding) 감성 모델의 정확도는 0.924로 머신러닝 모델과 거의 같다. …
채널 B :
  [F30] 유사도 0.52 | 이탈 모델 학습 대상은 2018-03~05에 판매한 판매자 1,567명이고, 이 중 377…
  [F31] 유사도 0.51 | 이탈 모델(로지스틱 회귀 + balanced)의 test 성능은 정밀도 0.43, 재현율 …
  [M09] 유사도 0.47 | M09 판매자 이탈 모델 (part9_seller_churn.pkl)
하는 일: 최근 판매…
  [M10] 유사도 0.42 | M10 분석 결과를 전략으로
재구매 3%라 성장이 신규 고객에만 의존한다 → 상품 추천(M…
채널 C :
  [F30] 유사도 0.52 | 이탈 모델 학습 대상은 2018-03~05에 판매한 판매자 1,567명이고, 이 중 377…
  [F31] 유사도 0.51 | 이탈 모델(로지스틱 회귀 + balanced)의 test 성능은 정밀도 0.43, 재현율 …
  [M09] 유사도 0.47 | M09 판매자 이탈 모델 (part9_seller_churn.pkl)
하는 일: 최근 판매…
  [M10] 유사도 0.42 | M10 분석 결과를 전략으로
재구매 3%라 성장이 신규 고객에만 의존한다 → 상품 추천(M…


**확인할 것**: 채널 A에는 모델 설명이 없으니 '이탈'이라는 말이 들어간 **숫자 사실**만 찾아옵니다. 채널 B는 `M09`(이탈 모델 설명)를 찾아옵니다.

## ④ 지침(프롬프트) + 답하기 + 근거 ID 점검

In [10]:
# 셀 7. 지침 + 질문 함수
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

NOT_FOUND = "제공된 자료로는 확인할 수 없습니다."
prompt = ChatPromptTemplate.from_messages([
    ("system",
     "너는 Olist 분석 결과 안내 챗봇이야.\n"
     "- 아래 [근거]에 있는 내용으로만 답해.\n"
     "- 문장마다 끝에 근거 ID를 [F01]처럼 붙여.\n"
     "- 규정을 인용할 때는 몇 조인지 함께 써.\n"
     "- 숫자는 근거의 숫자를 그대로 써. 직접 계산했다면 '(계산)'이라고 밝혀.\n"
     f"- [근거]로 답할 수 없으면 '{NOT_FOUND}'라고만 답해.\n"
     "- 한국어로 3문장 이내로 답해."),
    ("human", "[근거]\n{context}\n\n[질문]\n{question}"),
])
llm = ChatOpenAI(model="gpt-5.4-mini")
chain = prompt | llm | StrOutputParser()

def ask(question, channel, k=4):
    found = stores[channel].similarity_search(question, k=k)                    # ① 찾기
    context = "\n".join(f"({d.metadata['id']}) {d.page_content}" for d in found)
    answer = chain.invoke({"context": context, "question": question})           # ② 답하기
    found_ids = {d.metadata["id"] for d in found}
    cited = set(re.findall(r"\[([FMR]\d{2})\]", answer))                          # ③ 답에 쓴 ID
    fake = sorted(cited - found_ids)                                            #    찾지 않은 ID
    return answer, sorted(found_ids), fake

answer, found, fake = ask("늦게 도착한 주문의 리뷰 점수는?", "A")
print(answer)
print("찾아온 근거 :", found, "| 찾지 않은 ID :", fake or "없음")

늦게 도착한 주문의 평균 리뷰 점수는 2.57점입니다. [F14]
찾아온 근거 : ['F13', 'F14', 'F15', 'F16'] | 찾지 않은 ID : 없음


`fake`가 비어 있어야 정상입니다. 지침(부탁)만 믿지 않고 **코드로 한 번 더 확인**하는 안전장치입니다.

## ⑤ 세 채널 비교 : 같은 질문 7개

In [11]:
# 셀 8. 같은 질문을 A · B · C 세 채널에 던지기
questions = [
    "늦게 도착한 주문의 리뷰 점수는?",                 # 자료에 있음 (숫자)
    "재구매 고객은 몇 %야?",                          # 자료에 있음 (숫자)
    "판매자 유형은 몇 가지이고 어떻게 나눴어?",         # A는 숫자만, B는 방법까지
    "이탈 모델은 어떤 숫자를 보고 판단해?",             # A는 답 못 함, B·C는 답함
    "배송이 3일 늦으면 고객은 무엇을 받을 수 있어?",     # C만 답함 (규정 제6조)
    "저평점 위험 판매자는 어떤 조치를 받아?",            # B는 관리 방향, C는 규정 제13조까지
    "2019년 매출은 얼마야?",                          # 자료에 없음 → 모두 '확인 불가'
]
rows = []
for q in questions:
    row, fakes = {"질문": q}, []
    for ch, name in [("A", "A. 사실표"), ("B", "B. +설명서"), ("C", "C. +규정집")]:
        answer, _, fake = ask(q, ch)
        row[name] = answer
        fakes += fake
    row["가짜 ID"] = fakes
    rows.append(row)

pd.set_option("display.max_colwidth", None)
display(pd.DataFrame(rows))

,질문,A. 사실표,B. +설명서,C. +규정집,가짜 ID
0,늦게 도착한 주문의 리뷰 점수는?,늦게 도착한 주문의 평균 리뷰 점수는 2.57점입니다.[F14],"늦게 도착한 주문의 평균 리뷰 점수는 2.57점입니다. [F14] \n비교하면 제때 도착한 주문은 4.30점, 평균 리뷰 점수는 4.16점입니다. [F14] [F13]",늦게 도착한 주문의 평균 리뷰 점수는 2.57점입니다. [F14],[]
1,재구매 고객은 몇 %야?,재구매 고객은 3.0%입니다. [F07],재구매 고객은 3.0%입니다. [F07],재구매 고객은 3.0%입니다. [F07],[]
2,판매자 유형은 몇 가지이고 어떻게 나눴어?,"판매자 유형은 4가지이며, 핵심 우량 876명, 신규 성장 947명, 휴면 461명, 저평점 위험 218명으로 나눴습니다. [F27] \n이는 2018-06-01 이전 판매 기록을 기준으로 구분한 것입니다. [F27]","판매자 유형은 4가지이며, 핵심 우량 876명, 신규 성장 947명, 휴면 461명, 저평점 위험 218명으로 나눴습니다. [F27] \n이 분류는 판매자 숫자 5개(주문 건수, 판매액, 마지막 판매 후 지난 날 수, 첫 판매 후 지난 날 수, 평균 리뷰 점수)를 사용한 K-Means 군집으로, 비슷한 판매자끼리 4묶음으로 나눈 것입니다. [M08]","판매자 유형은 4가지이며, 핵심 우량·신규 성장·휴면·저평점 위험으로 나뉩니다. [F27] \nM08 모델은 판매자 숫자 5개(주문 건수, 판매액, 마지막 판매 후 지난 날 수, 첫 판매 후 지난 날 수, 평균 리뷰 점수)를 사용해 K-Means 군집으로 비슷한 판매자를 4묶음으로 나누고, 묶음 번호를 유형 이름으로 연결합니다. [M08]",[]
3,이탈 모델은 어떤 숫자를 보고 판단해?,"이탈 모델은 test 판매자 314명 기준으로 정밀도 0.43, 재현율 0.75, F1 0.55, AUC 0.81을 보고 판단합니다. 비교 규칙인 최근 30일 판매 없음의 F1은 0.53입니다. [F31]","이탈 모델은 마지막 판매 후 일수, 첫 판매 후 일수, 판매한 달 수, 주문 수·판매액·주문당 금액(로그), 평균 배송 일수, 도착 지연율, 출고 지연율, 평균 리뷰, 1~2점 리뷰 비율, 최근 3개월·그 전 3개월 주문 수(로그), 주문 증감의 14개 숫자를 보고 판단합니다. [M09] \n이 숫자들은 주문 기록 전체를 모아 계산해야 해서 판매자 표(seller_table.csv)로 미리 만들어 두고 웹앱은 그 표를 읽어 예측합니다. [M09]","이탈 모델은 마지막 판매 후 일수, 첫 판매 후 일수, 판매한 달 수, 주문 수·판매액·주문당 금액(로그), 평균 배송 일수, 도착 지연율, 출고 지연율, 평균 리뷰, 1~2점 리뷰 비율, 최근 3개월·그 전 3개월 주문 수(로그), 주문 증감의 14개 숫자를 보고 판단합니다. [M09] \n이 숫자들은 주문 기록 전체를 모아 계산해야 해서, seller_table.csv로 미리 만들어 두고 웹앱은 그 표를 읽어 예측합니다. [M09]",[]
4,배송이 3일 늦으면 고객은 무엇을 받을 수 있어?,제공된 자료로는 확인할 수 없습니다.,제공된 자료로는 확인할 수 없습니다.,배송 예정일보다 3일 이상 늦게 도착한 주문에서 고객이 요청하면 배송비를 환불받을 수 있습니다. [F11][R06 제6조] 비용은 지연 책임이 있는 쪽(판매자 또는 택배사)이 부담합니다. [R06 제6조],[]
5,저평점 위험 판매자는 어떤 조치를 받아?,제공된 자료로는 확인할 수 없습니다.,"저평점 위험 판매자는 평균 배송일이 약 22일, 도착 지연율 약 31%, 출고 지연율 약 33%, 1~2점 리뷰 비율이 80%를 넘는 유형으로 분류되며, 이탈 확률 0.5 이상이면 위험으로 봅니다. [F29][M09] \n다만 자료에는 이 유형에 대해 어떤 조치를 받는지는 제시되어 있지 않습니다. [F27][F29][M09]",저평점 위험 판매자는 제13조에 따라 출고 기한 준수 계획서를 14일 안에 제출해야 합니다. [R13] \n또한 다음 분기에도 저평점 위험 등급이면 신규 상품 등록이 제한됩니다. [R13],[]
6,2019년 매출은 얼마야?,제공된 자료로는 확인할 수 없습니다. [F02],"제공된 자료로는 확인할 수 없습니다. [F02] \n근거에는 2018-08-23 기준 매출과 원본 데이터 규모만 있고, 2019년 매출 수치는 없습니다. [F02][F19]","제공된 자료로는 확인할 수 없습니다. [F02] \n근거에는 2018-08-23 기준 분석 매출 합계만 있고, 2019년 매출은 제시되어 있지 않습니다. [F02]",[]


### 결과 읽는 법

| 질문 종류 | A (사실표) | B (+ 설명서) | C (+ 규정집) |
|---|---|---|---|
| 분석 숫자 | 근거 ID를 붙여 답 | 같은 답 | 같은 답 |
| 모델이 하는 일 | '확인할 수 없음' 또는 숫자 사실만 | 설명서 `M` 근거로 답 | 같은 답 |
| 규정 (배송 보상·등급별 조치) | '확인할 수 없음' | 관리 방향까지만 | 규정 `R` 근거로 몇 조인지 답 |
| 자료에 없는 것 (2019년) | '확인할 수 없음' | '확인할 수 없음' | '확인할 수 없음' |

→ 챗봇이 아는 범위 = **책장에 넣은 자료의 범위**입니다. 모르는 것을 지어내지 않게 하는 것이 RAG의 핵심입니다.

## ⑥ 챗봇 검수표

| 점검 질문 | 확인 방법 | 문제일 때 고칠 곳 |
|---|---|---|
| 맞는 조각을 찾아왔나? | 셀 6처럼 찾아온 조각 출력 | 사실표 문장, k |
| 조각에 없는 말을 했나? | 답과 사실표를 한 줄씩 대조 | 지침 |
| 근거 ID가 붙었나? 가짜 ID는? | `fake` 확인 | 지침, 점검 코드 |
| 모르면 모른다고 했나? | 자료 밖 질문 던지기 | 지침 |

In [12]:
# 셀 9. (검산) 사실표 숫자를 원본 데이터로 다시 계산해 보기
df = pd.read_csv(f"{base_path}/data/olist_master_data.csv", parse_dates=[
    "order_purchase_timestamp", "order_delivered_customer_date", "order_estimated_delivery_date"])
o = df.drop_duplicates("order_id")
o = o[(o["order_status"] == "delivered") & o["order_delivered_customer_date"].notna()
      & (o["order_purchase_timestamp"] >= "2017-01-05") & (o["order_purchase_timestamp"] < "2018-08-30")]
late = o["order_delivered_customer_date"] > o["order_estimated_delivery_date"]
repeat = (o.groupby("customer_unique_id")["order_id"].count() >= 2).mean()

print(f"F01 주문 수   : {len(o):,}건")
print(f"F12 지연율    : {late.mean():.1%}")
print(f"F07 재구매    : {repeat:.1%}")
print(f"F14 리뷰      : 제때 {o.loc[~late, 'review_score'].mean():.2f}점 / 지연 {o.loc[late, 'review_score'].mean():.2f}점")

F01 주문 수   : 96,203건
F12 지연율    : 8.1%
F07 재구매    : 3.0%
F14 리뷰      : 제때 4.30점 / 지연 2.57점


사실표의 숫자(96,203건 · 8.1% · 3.0% · 4.30점/2.57점)와 같으면, 챗봇이 쓰는 근거를 **사람이 다시 확인한 것**입니다.

## 직접 해보기

| 단계 | 미션 |
|---|---|
| 기초 | 내가 만든 질문 3개를 세 채널에 던지고, 답에 붙은 ID를 사실표에서 찾아 대조 |
| 기본 | `k`를 2와 8로 바꿔 보고, 찾아오는 조각과 답이 어떻게 달라지는지 비교 |
| 심화 | 사실표에 내가 찾은 사실 3줄(F36~F38)을 추가하고, 그 사실을 묻는 질문에 답하는지 확인 |
| 도전 | 공개된 기관 규정이나 이용약관 PDF 하나로 셀 4-2를 다시 실행해 나만의 규정 챗봇 만들기 |

> 👉 같은 챗봇을 **Streamlit 대시보드의 '분석 Q&A' 메뉴**에서도 쓸 수 있습니다. (`utils/rag.py`가 이 노트북의 셀 3~7과 같은 코드)